In [8]:
vllm serve "LORA_THE_BEST" \
  --host 0.0.0.0 \
  --port 8001 \
  --api-key 7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51 \
  --served-model-name LORA_THE_BEST \
  --max-model-len 1600 \
  --max-num-seqs 4

SyntaxError: invalid decimal literal (593931465.py, line 4)

In [9]:
from openai import OpenAI
import time
client = OpenAI(
    base_url="http://localhost:8001/v1",
    api_key="7b956701760dccad68ef20684d635bd3ee82e2e8fa6287f048b572169f178d51"
)
from gpu_monitor import *
import pandas as pd
from tqdm import tqdm

In [10]:
start_time = time.perf_counter()
response = client.chat.completions.create(
    model="LORA_THE_BEST",
    messages=[{"role": "user", "content": "Мой кариотип 47,XX,t(5;10)(q12;p22). Что это значит?"}],
    max_tokens=1600,
    temperature=0.0
    
)

end_time = time.perf_counter()

# Извлекаем текст ответа
answer = response.choices[0].message.content

print("Ответ модели:")
print(answer)
print(f"\nВремя выполнения: {end_time - start_time:.3f} секунд")

Ответ модели:
Тип перестройки: сбалансированная реципрокная транслокация

1. Характеристика перестройки
У пациента выявлена реципрокная транслокация между хромосомами 5 и 10 с точками разрыва в областях q12 и p22 соответственно.
Это означает, что фрагмент хромосомы 5 (область 5q12) обменялся местами с фрагментом хромосомы 10 (область 10p22).
Поскольку материал не утрачен и не продублирован, кариотип сбалансированный — общее количество и состав генетического материала сохранены.

2. Фенотип носителя
У лиц со сбалансированными реципрокными транслокациями физическое и умственное развитие, как правило, нормальное.
Проблемы чаще проявляются в репродуктивной сфере (бесплодие/невынашивание), а не как системные клинические симптомы.

3. Репродуктивные последствия
У мужчин
• Возможна нормальная фертильность, однако часть сперматозоидов будет содержать несбалансированные хромосомные наборы из-за нарушенного спаривания хромосом в мейозе (формирование квадривалента).
• Возможны:
o повышенная доля 

In [11]:
df = pd.read_excel("Q_F_T.xlsx")
df

,Promts,References
0,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...
1,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн..."
2,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом..."
3,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ..."
4,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес..."
...,...,...
100,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...
101,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома..."
102,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а..."
103,"Мой кариотип 46,XX,del(3)(q29q29). Что это зна...","Кариотип: 46,XX,del(3)(q29q29)\nТип хромосомно..."


In [16]:
def api_ai_answer(prompt):
    monitor = GPUMonitor(device_id=0)
    monitor.start()
    
    start_time = time.perf_counter()
    
    response = client.chat.completions.create(
        model="LORA_THE_BEST",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1600,
        temperature=0.0
    )
    
    end_time = time.perf_counter()
    # Извлекаем текст ответа
    answer = response.choices[0].message.content

    
    avg_power, max_power, avg_util = monitor.stop()
    response_time = end_time - start_time
    completion_tokens = response.usage.completion_tokens
    tokens_per_second = completion_tokens / response_time if response_time > 0 else 0
    tokens_per_sec_per_watt = tokens_per_second / avg_power if avg_power > 0 else 0
    
    return {
        'answer': answer,
        'response_time': response_time,
        'completion_tokens': completion_tokens,
        'tokens_per_second': tokens_per_second,
        'avg_power_watts': avg_power,
        'max_power_watts': max_power,
        'avg_gpu_util': avg_util,
        'tokens_per_sec_per_watt': tokens_per_sec_per_watt
    }

In [17]:
api_ai_answer("Мой кариотип 47,XX,+i(12)(p10). Что это значит?")

{'answer': 'Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома короткого плеча 12 (i(12p))\nГруппа: синдром Паллаистера–Киллиана\n\n1. Суть\nДополнительная изохромосома 12p → тетрасомия 12p (обычно мозаичная).\n\n2. Клиника\n• выраженная задержка развития\n• гипотония\n• характерные лицевые особенности\n• врождённые пороки\n\n3. Особенности\nЧасто не выявляется в крови, требуется исследование фибробластов.\n\n4. Заключение\ni(12p) ассоциирована с синдромом Паллаистера–Киллиана, тяжесть зависит от уровня мозаицизма.',
 'response_time': 2.0339739880000707,
 'completion_tokens': 198,
 'tokens_per_second': 97.34637766665142,
 'avg_power_watts': 270.13976190476194,
 'max_power_watts': 486.495,
 'avg_gpu_util': 71.42857142857143,
 'tokens_per_sec_per_watt': 0.36035560622493995}

In [7]:
#df['answer'], df['response_time'] = zip(*df['Promts'].apply(lambda x: api_ai_answer(x)))

In [18]:
outputs = []
for k in tqdm(df['Promts'], desc = "Обработка запросов"):
    #print(k)
    outputs.append(api_ai_answer(k))
    

Обработка запросов: 100%|██████████| 105/105 [11:04<00:00,  6.33s/it]


In [20]:
answers = [item['answer'] for item in outputs]
response_times = [item['response_time'] for item in outputs]
completion_tokens = [item['completion_tokens'] for item in outputs]
tokens_per_second = [item['tokens_per_second'] for item in outputs]
avg_power = [item['avg_power_watts'] for item in outputs]
max_power = [item['max_power_watts'] for item in outputs]
avg_gpu_util = [item['avg_gpu_util'] for item in outputs]
tps_per_watt = [item['tokens_per_sec_per_watt'] for item in outputs]

In [21]:
df["Answers"] = answers
df["Response times"] = response_times
df["Completion tokens"] = completion_tokens
df["Tokens per second"] = tokens_per_second
df["Avg Power (W)"] = avg_power
df["Max Power (W)"] = max_power
df["Avg GPU Util (%)"] = avg_gpu_util
df["Tokens/s per Watt"] = tps_per_watt


In [22]:
df.to_excel("VLLM_TEST.xlsx", index = False)
df

,Promts,References,Answers,Response times,Completion tokens,Tokens per second,Avg Power (W),Max Power (W),Avg GPU Util (%),Tokens/s per Watt
0,"Мой кариотип 46,XX,t(9;13)(p11;q21). Что это з...",Тип перестройки: сбалансированная реципрокная ...,Тип перестройки: сбалансированная реципрокная ...,11.384447,1108,97.325764,432.893570,486.187,94.736842,0.224826
1,"Мой кариотип 46,XX,inv(17)(q32q33). Что это зн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн...","Кариотип: 46,XX,inv(17)(q32q33)\nТип хромосомн...",7.769514,756,97.303390,469.995397,470.173,100.000000,0.207031
2,"Мой кариотип 47,XX,+8[10]/46,XX[15]. Что это з...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом...","Кариотип: 47,XX,+8[10]/46,XX[15]\nТип хромосом...",6.034162,587,97.279456,469.992148,470.119,100.000000,0.206981
3,"Мой кариотип 49,XXXXY. Что это значит?","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ...","Кариотип: 49,XXXXY\nТип хромосомной аномалии: ...",1.123177,109,97.046184,454.582083,480.399,65.500000,0.213484
4,"Мой кариотип 45,XY,der(22;22)(q10;q10). Что эт...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес...","Кариотип: 45,XY,der(22;22)(q10;q10)\nТип перес...",10.861995,1054,97.035580,469.198587,484.252,96.587156,0.206811
...,...,...,...,...,...,...,...,...,...,...
100,"Мой кариотип 45,XY,der(21;22)(q10;q10). Что эт...",Тип перестройки: сбалансированная Робертсоновс...,Тип перестройки: сбалансированная Робертсоновс...,8.100445,779,96.167556,470.577543,479.512,100.000000,0.204361
101,"Мой кариотип 47,XX,+i(12)(p10). Что это значит?","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома...","Кариотип: 47,XX,+i(12)(p10)\nТип: изохромосома...",2.060536,198,96.091488,469.740000,470.352,97.142857,0.204563
102,"Мой кариотип 46,XX,del(8)(p22). Что это значит?","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а...","Кариотип: 46,XX,del(8)(p22)\nТип хромосомной а...",3.833426,369,96.258554,470.351359,483.280,95.230769,0.204652
103,"Мой кариотип 46,XX,del(3)(q29q29). Что это зна...","Кариотип: 46,XX,del(3)(q29q29)\nТип хромосомно...","Кариотип: 46,XX,del(3)(q29q29)\nТип хромосомно...",3.002003,289,96.269071,466.201000,472.822,91.400000,0.206497
